# Ship 2: IBM Granite Mirror RAG Pipeline

**Based on:** IBM SkillsBuild Lab — Build a RAG Pattern with LangChain (08/08/2026)
**Adapted for:** A Mirror of My Becoming — Sovereign personal archive
**Author:** Evelyn Caro (@qaevelyn)
**Date:** 08/12/2026

## About This Notebook

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline for **A Mirror of My Becoming** — a sovereign personal archive.

**What this notebook does:**
1. Loads Mirror documents (PDFs, Markdown, text files)
2. Splits them into chunks
3. Creates embeddings using IBM Granite (via Ollama)
4. Stores them in a vector database (ChromaDB)
5. Queries the archive and generates responses

**Based on:** IBM SkillsBuild Lab — Build a RAG Pattern with LangChain
**Adapted by:** Evelyn Caro (@qaevelyn)
**Mission:** Sovereign data ownership for Foundational Black Americans and Freedmen

In [ ]:
# ============================================================
# CELL 1: SET UP ENVIRONMENT AND LOAD DEPENDENCIES
# ============================================================

get_ipython().system('pip install langchain langchain-community langchain-text-splitters langchain-ollama chromadb langchain-chroma pypdf')

print("✅ Dependencies installed")

In [ ]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================

import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma

print("✅ Imports ready")

In [1]:
# ============================================================
# CELL 3: SET UP EMBEDDINGS
# ============================================================

from langchain_ollama import OllamaEmbeddings

# Use nomic-embed-text for embeddings (supports Ollama embeddings)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Tokenizer placeholder (not needed for Ollama)
embeddings_tokenizer = None

print("✅ Embeddings ready with nomic-embed-text")

✅ Embeddings ready with nomic-embed-text


In [2]:
# ============================================================
# CELL 4: SET UP MODEL (IBM Granite)
# ============================================================

from langchain_ollama import ChatOllama, OllamaEmbeddings

# Use IBM Granite via Ollama (no stop parameter)
llm = ChatOllama(
    model="granite4.1:3b",
    temperature=0.7,
)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
)

print("✅ Model and embeddings ready")
print("✅ LLM ready with granite4.1:3b")

✅ Model and embeddings ready
✅ LLM ready with granite4.1:3b


In [4]:
# ============================================================
# CELL 5: CREATE VECTOR STORE
# ============================================================

from langchain_chroma import Chroma

# Create vector store from the text chunks
vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✅ Vector store created with", len(texts), "chunks")

# Create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# Run a similarity search
query = "Why are you interested in this program?"
results = retriever.invoke(query)

print(f"Found {len(results)} relevant chunks")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")

✅ Vector store created with 102 chunks
Found 4 relevant chunks

--- Chunk 1 ---
- **Answer:** No — not enrolled, not eligible
- **Suggestion:** Use alumni access or free tier
### 1:58 PM — Lab Options Presented
- **Options:** Google Colab, Local, IBM watsonx
- **Decision:** Choos...

--- Chunk 2 ---
- **Answer:** No — not enrolled, not eligible
- **Suggestion:** Use alumni access or free tier
### 1:58 PM — Lab Options Presented
- **Options:** Google Colab, Local, IBM watsonx
- **Decision:** Choos...

--- Chunk 3 ---
- **Answer:** No — not enrolled, not eligible
- **Suggestion:** Use alumni access or free tier
### 1:58 PM — Lab Options Presented
- **Options:** Google Colab, Local, IBM watsonx
- **Decision:** Choos...

--- Chunk 4 ---
- **Email:** `evelynacaro@aol.com`
- **Canvas URL:** `https://perscholas.instructure.com/courses/3569`
### 10:50 AM — Syllabus Analysis
- **Voucher requirements:** 5 practice tests (≥70/90), 2 qualifi...


In [3]:
# ============================================================
# CELL 6: LOAD AND CHUNK DOCUMENT (Optimized)
# ============================================================

import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# Define the file path
file_path = "/Users/evelyn/Documents/Mirror-Project/MIRROR_LOG.md"

if os.path.exists(file_path):
    print(f"✅ Loading: {file_path}")
    loader = TextLoader(file_path)
    documents = loader.load()
else:
    print(f"❌ File not found: {file_path}")
    import wget
    url = "https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/foundation_models/state_of_the_union.txt"
    filename = "state_of_the_union.txt"
    if not os.path.exists(filename):
        wget.download(url, out=filename)
    loader = TextLoader(filename)
    documents = loader.load()

# Split into chunks — larger chunks = faster
text_splitter = CharacterTextSplitter(
    chunk_size=1000,   # Increased from 500
    chunk_overlap=100, # Increased from 50
    separator="\n",
)

texts = text_splitter.split_documents(documents)

print(f"✅ Created {len(texts)} chunks")

/var/folders/4l/746jmy4j7b3711l8cgn7h1fm0000gn/T/ipykernel_8691/2843833093.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/evelyn/Documents/Mirror-Project/granite-workshop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Loading: /Users/evelyn/Documents/Mirror-Project/MIRROR_LOG.md
✅ Created 102 chunks


In [5]:
# ============================================================
# CELL 7: DEFINE THE RAG TOOL
# ============================================================

from langchain_classic.tools import tool

# Create retriever (if not already defined)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

@tool
def get_granite_context(question: str) -> str:
    """Retrieve relevant context from the Mirror archive."""
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    return context

tools = [get_granite_context]

In [6]:
# ============================================================
# CELL 8: SET UP THE AGENT PROMPT
# ============================================================

from langchain_classic.tools.render import render_text_description_and_args
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are a helpful AI assistant. You have access to a tool that can retrieve relevant context from a personal archive.

Use the tool to answer questions when you don't have the information.

Provide only ONE action per JSON blob.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
]).partial(
    tools=render_text_description_and_args(tools),
    tool_names=", ".join([t.name for t in tools]),
)

In [7]:
# ============================================================
# CELL 9: SET UP AGENT MEMORY AND CHAIN
# ============================================================

from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant. You have access to the following tools:

{tools}

Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=3,  # Limit to 3 loops to prevent infinite
)

In [8]:
# ============================================================
# CELL 10: RUN THE AGENTIC RAG QUERY
# ============================================================

response = agent_executor.invoke({
    "input": "What is the main topic of the document?"
})

print("=" * 60)
print("QUESTION: What is the main topic of the document?")
print("=" * 60)
print(response["output"])



> Entering new AgentExecutor chain...
Question: What is the main topic of the document?
Thought: To determine the main topic of the document, I should retrieve relevant context from the Mirror archive using the get_granite_context tool.
Action: get_granite_context
Action Input: (I would need a specific question or prompt to feed into the tool, but since no context is provided, I'll assume we need general guidance)- **Query 3:** "What was IBM's role or usage of watsonx?" → Retrieved relevant documents
### 12:44 PM — Certificate Earned
- **Certificate:** Lab: Build a LangChain Agentic RAG System with IBM Granite
- **Course ID:** ALM-COURSE_4061305
- **Date:** 08/10/2026
### 12:50 PM — Notebook Pushed to GitHub
- **File:** `Agentic_RAG_IBM_Granite.ipynb`
- **Commit:** "Ship 2: Agentic RAG with IBM Granite - completed 08/10/2026"
### 12:51 PM — Certificate Link Logged
### 1:02 PM — Three Ships Summary
- **Ship 1:** AWS + DeepSeek (Standard RAG)
- **Ship 2:** IBM Granite (Standard RAG + A